kpi computation and exports


In [1]:
import pandas as pd
import json


In [2]:
df = pd.read_csv('../data/processed/nyc_parking_clean.csv')


In [3]:
kpis = {
    'total_tickets': len(df),
    'total_revenue': float(df['Fine_Amount'].sum()),
    'avg_fine_per_ticket': float(df['Fine_Amount'].mean().round(2)),
    'outofstate_pct': float((df['Is_OutOfState'].mean()*100).round(1)),
    'repeat_offender_pct': float((df['Is_Repeat_Offender'].mean()*100).round(1)),
    'safety_violation_pct': float((df['Violation_Severity'] == 'Safety-Critical').mean()*100),
    'avoidable_pct': float((df['Is_Avoidable'].mean()*100).round(1)),
    'unique_vehicles': int(df['Plate_ID'].nunique()),
    'peak_month': str(df.groupby('Month_Name')['Summons_Number'].count().idxmax()),
    'top_borough_revenue': str(df.groupby('Violation_County')['Fine_Amount'].sum().idxmax())
}


In [4]:
with open('../docs/kpi_summary.json', 'w') as f:
    json.dump(kpis, f, indent=2)


In [5]:
borough = df.groupby('Violation_County').agg(
    ticket_count=('Summons_Number', 'count'),
    total_revenue=('Fine_Amount', 'sum'),
    avg_fine=('Fine_Amount', 'mean')
).reset_index()
borough.to_csv('../data/processed/tableau_borough.csv', index=False)


In [6]:
monthly = df.groupby('Month_Name').agg(
    ticket_count=('Summons_Number', 'count')
).reset_index()
monthly.to_csv('../data/processed/tableau_monthly.csv', index=False)


In [7]:
streets = df.groupby('Street_Name').agg(
    ticket_count=('Summons_Number', 'count'),
    total_revenue=('Fine_Amount', 'sum')
).reset_index()
streets.to_csv('../data/processed/tableau_streets.csv', index=False)
